# GraphMS-Net — Segmentation Validation Audit

**Purpose:** rapid, no-retraining validation of the frozen GraphMS v3.5.1 Hybrid segmentation system.

This notebook does **not** train, tune, select thresholds, or alter the frozen pipeline. It uses the committed Stage16 per-case table and the already-generated final hybrid masks to produce reviewer-facing qualitative evidence: **FLAIR | expert ground truth | final prediction | TP/FP/FN error map** for representative best, median, and worst development cases.

Claim scope remains **five-fold development cross-validation**; this is not external clinical validation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO = '/content/GraphMS-Net'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git','clone','-q','https://github.com/sath17-o/GraphMS-Net.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'pull','-q','--ff-only'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','nibabel'], check=True)
print('Repository and dependencies ready.')


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

METRICS = Path(REPO) / 'results/stage16/STAGE16_PER_CASE_METRICS.csv'
df = pd.read_csv(METRICS)
assert len(df) == 93, f'Expected 93 final development cases, found {len(df)}'
print('Cases:', len(df), '| folds:', sorted(df.fold.unique().tolist()))
print('DSC range:', float(df.DSC.min()), 'to', float(df.DSC.max()))


## Representative-case selection

Cases are selected **deterministically from the frozen Stage16 DSC column**. No case is hand-picked by visual appearance. The median case is the row closest to the cohort median DSC.


In [ ]:
best = df.loc[df.DSC.idxmax()]
worst = df.loc[df.DSC.idxmin()]
median_value = float(df.DSC.median())
median = df.loc[(df.DSC - median_value).abs().idxmin()]
selected = pd.DataFrame([best, median, worst], index=['BEST','MEDIAN','WORST'])
display(selected[['case','fold','DSC','IoU','Sensitivity','Specificity','HD95_mm','TP','FP','FN']])


In [ ]:
def load_bool(path):
    return np.asarray(nib.load(str(path)).dataobj) > 0

def robust_flair(path):
    x = np.asarray(nib.load(str(path)).dataobj, dtype=np.float32)
    finite = np.isfinite(x)
    if finite.any():
        lo, hi = np.percentile(x[finite], [1, 99])
        x = np.clip((x-lo)/(hi-lo+1e-8), 0, 1)
    return x

def choose_slice(gt, pred):
    burden = np.sum(gt | pred, axis=(0,1))
    return int(np.argmax(burden))

def show_case(row, label):
    case = row['case']
    gt_path = Path(row['gt_path'])
    pred_path = Path(row['mask_path'])
    flair_path = gt_path.parent.parent / 'imagesTr' / f'{case}_0000.nii.gz'
    for p in [gt_path, pred_path, flair_path]:
        assert p.exists(), f'Missing required file: {p}'

    gt = load_bool(gt_path)
    pred = load_bool(pred_path)
    flair = robust_flair(flair_path)
    assert gt.shape == pred.shape == flair.shape, (gt.shape, pred.shape, flair.shape)
    z = choose_slice(gt, pred)

    tp = gt & pred
    fp = (~gt) & pred
    fn = gt & (~pred)
    err = np.zeros(gt.shape, dtype=np.uint8)
    err[tp] = 1; err[fp] = 2; err[fn] = 3

    fig, ax = plt.subplots(1,4,figsize=(16,4), constrained_layout=True)
    for a in ax:
        a.imshow(flair[:,:,z].T, cmap='gray', origin='lower')
        a.axis('off')
    ax[0].set_title(f'{label}: {case}\nFLAIR — slice {z}')
    ax[1].imshow(np.ma.masked_where(~gt[:,:,z].T, gt[:,:,z].T), cmap='autumn', alpha=.65, origin='lower')
    ax[1].set_title('Expert ground truth')
    ax[2].imshow(np.ma.masked_where(~pred[:,:,z].T, pred[:,:,z].T), cmap='winter', alpha=.65, origin='lower')
    ax[2].set_title('Frozen final prediction')
    cmap = ListedColormap(['black','lime','red','deepskyblue'])
    ax[3].imshow(np.ma.masked_where(err[:,:,z].T==0, err[:,:,z].T), cmap=cmap, vmin=0, vmax=3, alpha=.8, origin='lower')
    ax[3].set_title('Error map: TP green | FP red | FN blue')
    fig.suptitle(f"DSC {row['DSC']:.4f} | IoU {row['IoU']:.4f} | Sens {row['Sensitivity']:.4f} | Spec {row['Specificity']:.6f} | HD95 {row['HD95_mm']:.2f} mm", fontsize=11)
    out = Path('/content') / f"SEGMENTATION_{label}_{case}.png"
    fig.savefig(out, dpi=220, bbox_inches='tight')
    plt.show()
    return out

outputs=[]
for label, row in selected.iterrows():
    outputs.append(show_case(row, label))
print('Saved:', *outputs, sep='\n - ')


## Interpretation

The three panels expose both strengths and failure modes without changing the frozen system. **TP** shows correctly segmented lesion voxels, **FP** shows predicted lesion voxels absent from the expert mask, and **FN** shows expert lesion voxels missed by the model.

The final system remains: ResEncM-250 → graph construction → TrueGAT → CNN/GNN hybrid fusion → decoder/head → cross-fitted Stage11. Fold-specific Stage11 thresholds remain frozen (0.40 for folds 0/1/3/4; 0.45 for fold 2), with 26-connectivity and a 10-voxel minimum component size. No morphology is introduced by this audit.
